# A4 — cluster versus adjacent-normal epithelium

Compare epithelium to epithelium. Gate: proliferation genes up in every cluster.
If this gate fails, A5.1 reversal is omitted; A5.2 nearest lines still ship.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False

def cohort_ids():
    import pandas as pd
    assign = V3 / "cluster_assignments.parquet"
    if assign.is_file():
        return pd.read_parquet(assign)["patient_id"].astype(str).str[:12].unique().tolist()
    expr = INTERIM / "intrinsic_expression.parquet"
    if expr.is_file():
        return pd.read_parquet(expr).index.astype(str).str[:12].unique().tolist()
    return None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        ids = cohort_ids()
        if ids is not None:
            kwargs["sample_ids"] = ids
            kwargs.setdefault("n", len(ids))
            kwargs.setdefault("cohort", True)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import pandas as pd
from tcga_normals import (
    PROLIF_GENES, cluster_vs_normal_signatures, intrinsic_normal_epithelium,
    proliferation_gate, sample_type_from_barcode, split_tumour_normal,
)
expr_p = INTERIM / "intrinsic_expression.parquet"
deconv_p = INTERIM / "deconvolution_posterior.parquet"
used_real = False
if expr_p.is_file():
    expr = pd.read_parquet(expr_p)
    expr.index = expr.index.astype(str)
    tumours, normals = split_tumour_normal(expr.index)
    if len(normals) >= 10 and len(tumours) >= 20:
        used_real = True
        deconv = pd.read_parquet(deconv_p) if deconv_p.is_file() else None
        t_int = intrinsic_normal_epithelium(expr.loc[tumours], deconv.loc[tumours] if deconv is not None and len(set(tumours) & set(deconv.index)) else None)
        n_int = intrinsic_normal_epithelium(expr.loc[normals], deconv.loc[normals] if deconv is not None and len(set(normals) & set(deconv.index)) else None)
        preg = json.loads((REF / "preregistered_k.json").read_text())
        assign = pd.read_parquet(V3 / "cluster_assignments.parquet")
        sub = assign[(assign["method"]=="gmm") & (assign["covariance_type"]=="full") & (assign["k"]==preg.get("k"))]
        labels = sub.set_index("patient_id")["cluster"]
        t_int.index = t_int.index.astype(str)
        # align labels to tumour barcodes via patient id
        lab = []
        keep = []
        for idx in t_int.index:
            pid = idx[:12]
            if pid in labels.index:
                keep.append(idx)
                lab.append(int(labels.loc[pid] if not hasattr(labels.loc[pid], 'iloc') else labels.loc[pid].iloc[0]))
        stats_df, meta = cluster_vs_normal_signatures(t_int.loc[keep], n_int, pd.Series(lab, index=keep))
        sigs = {int(c): stats_df[stats_df.cluster==c].set_index("feature")["log2fc"] for c in stats_df.cluster.unique()}
        prolif = proliferation_gate(sigs)
        stats_df.to_parquet(V3 / "cluster_vs_normal_signature.parquet")
        (V3 / "normal_reference_v3.parquet").parent.mkdir(parents=True, exist_ok=True)
        n_int.to_parquet(V3 / "normal_reference_v3.parquet")
        (V3 / "a4_meta.json").write_text(json.dumps({**meta, **prolif, "passed": prolif["passed"]}, indent=2))

if not used_real:
    print("A4: real tumour/normal split too thin — not synthesizing. Gate will fail.")
    prolif = {"passed": False, "per_cluster_mean_logfc": {}}
    (V3 / "a4_meta.json").write_text(json.dumps({"passed": False, "note": "insufficient real normals; no synthetic fallback"}, indent=2))

print(prolif)


In [ ]:
a4 = json.loads((V3 / "a4_meta.json").read_text())
means = a4.get("per_cluster_mean_logfc") or {}
gate("NB_A4", "proliferation_up_vs_normal", int(bool(a4.get("passed"))), 1,
     note=f"per-cluster proliferation mean logFC: {means}")
